In [3]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# ── Read files ────────────────────────────────────────────────────────────────
roads   = pd.read_csv("_roads3.csv")
bridges = pd.read_excel("BMMS_overview.xlsx")

# ── Haversine distance (km) ───────────────────────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# ── Nearest point within threshold ───────────────────────────────────────────
def nearest(lat, lon, df, threshold):
    dists = np.array([haversine(lat, lon, r, c)
                      for r, c in zip(df["lat"], df["lon"])])
    i = dists.argmin()
    return (i, dists[i]) if dists[i] <= threshold else (None, None)

# ── Road length in km (sum of segment distances) ─────────────────────────────
def road_length(df):
    coords = df[["lat","lon"]].values
    return sum(haversine(coords[i,0], coords[i,1],
                         coords[i+1,0], coords[i+1,1])
               for i in range(len(coords)-1))

# ── Does a road intersect N1 or N2? (any point within 0.5 km) ────────────────
def intersects_main(group, main_df, threshold_km=0.5):
    for _, row in group.iterrows():
        i, dist = nearest(row["lat"], row["lon"], main_df, threshold_km)
        if i is not None:
            return True
    return False

# ── Select N1 and N2 ──────────────────────────────────────────────────────────
n1 = roads[roads["road"] == "N1"].reset_index(drop=True)
n2 = roads[roads["road"] == "N2"].reset_index(drop=True)

# ── Select side roads: N-roads longer than 25 km that intersect N1 or N2 ─────
n_roads = roads[roads["road"].str.match(r"^N\d+$") &
                ~roads["road"].isin(["N1", "N2"])].copy()

side_roads = []
for road_name, group in n_roads.groupby("road"):
    group = group.reset_index(drop=True)
    if road_length(group) > 25 and \
       (intersects_main(group, n1) or intersects_main(group, n2)):
        side_roads.append(group)

if side_roads:
    side = pd.concat(side_roads, ignore_index=True)
else:
    side = pd.DataFrame(columns=roads.columns)

print(f"Side roads selected (N-roads > 25 km intersecting N1 or N2): "
      f"{side['road'].unique().tolist() if not side.empty else []}")

# ── Preprocess a single road ──────────────────────────────────────────────────
def preprocess(df, road_name, is_main=False):
    df = df[["road","lrp","lat","lon","chainage"]].copy()

    # Merge bridge info
    b = bridges[bridges["road"] == road_name][
            ["LRPName","length","condition","constructionYear"]]
    df = df.merge(b, how="left", left_on="lrp", right_on="LRPName")

    # Assign model type
    df["model_type"] = np.select([df["length"].notna()], ["bridge"], default="link")

    if is_main:
        df.iloc[0,  df.columns.get_loc("model_type")] = "source"
        df.iloc[-1, df.columns.get_loc("model_type")] = "sink"
    else:
        # Side road endpoints are sourcesink (can generate and absorb traffic)
        df.iloc[0,  df.columns.get_loc("model_type")] = "sourcesink"
        df.iloc[-1, df.columns.get_loc("model_type")] = "sourcesink"

    # Fill missing lengths from chainage difference (m)
    df["length"] = df["length"].fillna(
        (df["chainage"].shift(-1) - df["chainage"]) * 1000)

    df["constructionYear"] = pd.to_numeric(df["constructionYear"], errors="coerce")
    df["quality_score"]    = df["condition"].map(
        {"A":4,"B":3,"C":2,"D":1}).fillna(0)

    df = df.rename(columns={"lrp": "name"})
    df = df.drop_duplicates("name", keep="last")
    return df.drop(columns=["LRPName","chainage"], errors="ignore").reset_index(drop=True)

n1_p   = preprocess(n1, "N1", is_main=True)
n2_p   = preprocess(n2, "N2", is_main=True)
side_p = pd.concat([preprocess(g.reset_index(drop=True), r, is_main=False)
                    for r, g in side.groupby("road")],
                   ignore_index=True) if not side.empty else pd.DataFrame()

# ── Find intersections between two road DataFrames ───────────────────────────
def intersections(df_a, df_b, la, lb, threshold_km=0.5):
    seen, rows = set(), []
    for _, rb in df_b.iterrows():
        i, dist = nearest(rb.lat, rb.lon, df_a, threshold_km)
        if i is None or i in seen:
            continue
        seen.add(i)
        ra = df_a.iloc[i]
        rows.append({
            "road":       la,
            "model_type": "intersection",
            "name":       f"INT_{la}_{lb}_{len(rows)+1}",
            "lat":        ra.lat,
            "lon":        ra.lon,
            "length":     0.0,
            "condition":  np.nan,
        })
    return pd.DataFrame(rows)

int_frames = [intersections(n1_p, n2_p, "N1", "N2")]

if not side_p.empty:
    int_frames.append(intersections(n1_p, side_p, "N1", "SIDE"))
    int_frames.append(intersections(n2_p, side_p, "N2", "SIDE"))

ints = pd.concat(int_frames, ignore_index=True)

# ── Assemble final output ─────────────────────────────────────────────────────
COLS = ["road","model_type","condition","name","lat","lon","length"]

frames = [n1_p, n2_p]

if not side_p.empty:
    frames.append(side_p)
frames.append(ints)

out = pd.concat(frames, ignore_index=True)

for c in COLS:
    if c not in out.columns:
        out[c] = np.nan

out = out[COLS].copy()
out.insert(1, "id", out.index + 1)
out[["lat","lon","length"]] = out[["lat","lon","length"]].round(6)

# Keep name only for edge pieces (source, sink, sourcesink) — clear all others
edge_types = {"source", "sink", "sourcesink"}
out.loc[~out["model_type"].isin(edge_types), "name"] = np.nan

out['model_type'].replace(['source', 'sink'], 'sourcesink', inplace=True)

out.to_csv("processed_data.csv", index=False)
print(f"\nDone: {len(out)} rows")
print(out["model_type"].value_counts().to_string())

# Print all edge nodes
sosink = out[out["model_type"].isin(edge_types)]
print(f"\nEdge nodes ({len(sosink)} rows):")
print(sosink.to_string(index=False))

Side roads selected (N-roads > 25 km intersecting N1 or N2): ['N102', 'N104', 'N105', 'N204', 'N207', 'N208']

Done: 3134 rows
model_type
link            2032
bridge          1072
sourcesink        16
intersection      14

Edge nodes (16 rows):
road   id model_type condition   name       lat       lon  length
  N1    1 sourcesink       NaN   LRPS 23.706028 90.443333   814.0
  N1 1339 sourcesink       NaN   LRPE 20.862917 92.298083     NaN
  N2 1340 sourcesink       NaN   LRPS 23.705917 90.521444   166.0
  N2 2225 sourcesink       NaN   LRPE 25.157056 92.017638     NaN
N102 2226 sourcesink       NaN   LRPS 23.478972 91.118194   329.0
N102 2444 sourcesink       NaN   LRPE 24.050611 91.114667     NaN
N104 2445 sourcesink       NaN   LRPS 23.009667 91.399416   445.0
N104 2567 sourcesink       NaN   LRPE 22.825749 91.101444     NaN
N105 2568 sourcesink       NaN   LRPS 23.690416 90.546611  1000.0
N105 2684 sourcesink       NaN LRP048 23.989527 90.358222     NaN
N204 2685 sourcesink       Na